In [1]:
# Import de
import numpy as np
import pandas as pd
import scipy.stats as stats
from pathlib import Path
import json

from support_functions import (
    conditional_sgm,
    convert_to_single_synteyes,
    create_retina_curvature,
    create_mgmm_data,
    nearest_psd,
    zernike_index
    )


## Model parameters

Parameters of the MGM and SGM creating the SyntEyes with retina radii.

In [2]:
# Original SyntEyes parameters
base_path = Path(__file__).parent if "__file__" in globals() else Path.cwd()
modeldata_path = base_path / "modeldata.json"
modeldata = json.loads(modeldata_path.read_text(encoding="utf-8"))

conv_ec_orig = np.array(modeldata["conv_ec_orig"])
avg_ec_orig = np.array(modeldata["avg_ec_orig"])
lens_za_orig = np.array(modeldata["lens_za_orig"])
weights_orig = np.array(modeldata["weights_orig"])
mu_orig = np.array(modeldata["mu_orig"])
cov_orig0 = np.array(modeldata["cov_orig0"])
cov_orig1 = np.array(modeldata["cov_orig1"])

cov_orig = np.zeros((2, np.shape(cov_orig0)[0], np.shape(cov_orig0)[1]))
cov_orig[0] = nearest_psd(cov_orig0)
cov_orig[1] = cov_orig1

# Retina model parameters
MU_AL_RADII = np.array([23.85, 11.89, 11.66, 10.55])
COV_AL_RADII = np.array(
    [
        [1.82, 0.56, 0.42, 0.81],
        [0.56, 0.36, 0.19, 0.18],
        [0.42, 0.19, 0.24, 0.16],
        [0.81, 0.18, 0.16, 0.48],
    ]
)

## Create n SyntEyes

In [3]:
# Create n 3D SynthEyes

# User input to set the amount of generated eyes
n = int(input("Number of SyntEyes: "))
print("There will be " + str(n) + " SyntEyes generated")

# Create the eigencornea data
eigen_data = create_mgmm_data(mu_orig[0],mu_orig[1],cov_orig0,cov_orig1,weights_orig[0],weights_orig[1],n)

# Use the eigencornea to create the SyntEyes without retina curvature information
synteyes_orig = pd.DataFrame([])
for i in range(len(eigen_data)):
    synteyes_orig_single = convert_to_single_synteyes(
        eigen_data[i], conv_ec_orig, avg_ec_orig, lens_za_orig
    )
    synteyes_orig = pd.concat([synteyes_orig, synteyes_orig_single],ignore_index=True)

# Add the retina curvature information
synteyes_data = create_retina_curvature(synteyes_orig, MU_AL_RADII, COV_AL_RADII)

print(synteyes_data)

There will be 10 SyntEyes generated
        CCT       ACD        LT  AxialLength         VD   RT        Rla  \
0  0.589329  2.692674  3.850647    25.125071  17.792421  0.2  11.316586   
1  0.489409  3.401578  3.865124    26.200501  18.244390  0.2  11.851734   
2  0.567856  2.872764  3.089215    23.308746  16.578911  0.2  12.859103   
3  0.589259  3.486178  3.737886    25.877205  17.863883  0.2  11.221415   
4  0.589182  2.994631  4.190336    25.673945  17.699796  0.2  10.347847   
5  0.551698  2.945117  3.955423    24.044684  16.392445  0.2  10.645869   
6  0.536739  3.195840  3.910582    23.262085  15.418923  0.2  10.480972   
7  0.559767  2.501301  4.382562    23.132004  15.488373  0.2   9.319380   
8  0.551596  2.776480  4.023449    23.409971  15.858446  0.2  10.599118   
9  0.542575  2.562475  4.274195    23.211531  15.632286  0.2   9.526043   

        Rlp     Qla  Qlp  ...  CorPostZ(8.0,-4.0)  CorPostZ(8.0,-2.0)  \
0 -7.436704 -3.1316   -1  ...            0.000010           -0.00

In [4]:
# Main Data to create a retina curvature corresponding to a provided Axial length
al = float(input("Which axial length (mm) does the eye have: "))
print(al)

# Calculate the conditional model belonging to the Axial Length
conditional_mean_sgm, conditional_cov_sgm = conditional_sgm(MU_AL_RADII, COV_AL_RADII, [0], al)
# Create a single retina curvature
rx, ry, rz = stats.multivariate_normal.rvs(
    mean=conditional_mean_sgm, cov=conditional_cov_sgm, size=1
)
print(rx, ry, rz)

24.0
11.444624926849265 11.109852412282471 11.440498224505474
